In [15]:
home = [
    [1, 1, 3, 1],
    [0, -1, 1, 1],
    [1, 0, 0, -1]
]

state = None
model = {
    "Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)
}
rules = []
action = None

last_pos = None
visited = [[0 for _ in range(len(home[0]))] for _ in range(len(home))]

In [16]:
def find_location(state):
    # Tìm vị trí hiện tại của máy hút bụi (ô có giá trị 2 hoặc 3)
    for i in range(len(state)):
        for j in range(len(state[0])):

            if state[i][j] == 2 or state[i][j] == 3:
                return i, j
            
    return None, None

In [17]:
def update_state(current_state, last_action, current_percept, world_model):
    # Cập nhật bản đồ bộ nhớ 'state' từ cảm biến 'percept'
    if current_state is None:
        current_state = [row[:] for row in current_percept]
    
    x, y = find_location(current_percept)
    if x is not None:
        current_state[x][y] = current_percept[x][y]
        
    return current_state

In [18]:
import random

def rule_match(current_state, last_action, current_pos, previous_pos):
    x, y = current_pos
    rows = len(current_state)
    cols = len(current_state[0])
    
    if current_state[x][y] == 3:
        return "Suck"
    
    # Chỉ giữ lại các hướng nằm trong biên và không phải vật cản
    possible_moves = []
    if x > 0 and current_state[x-1][y] != -1: possible_moves.append("Up")
    if x < rows - 1 and current_state[x+1][y] != -1: possible_moves.append("Down")
    if y > 0 and current_state[x][y-1] != -1: possible_moves.append("Left")
    if y < cols - 1 and current_state[x][y+1] != -1: possible_moves.append("Right")

    # Nếu không còn rác thì dừng
    if not any(1 in row or 3 in row for row in current_state):
        return "Noop"

    # Né hướng vừa bị kẹt
    if current_pos == previous_pos and last_action in possible_moves:
        possible_moves.remove(last_action)
    
    move_scores = {}
    for move in possible_moves:
        dx, dy = model[move]
        nx, ny = x + dx, y + dy
        move_scores[move] = visited[nx][ny] # Lấy số lần đã đi qua của ô kế tiếp
        
    # Tìm số lần ghé thăm nhỏ nhất trong các hướng có thể đi
    min_visit = min(move_scores.values())
    
    # Chỉ giữ lại các hướng có số lần ghé thăm bằng với min_visit
    best_moves = [move for move in possible_moves if move_scores[move] == min_visit]
    
    return random.choice(best_moves)

In [19]:
def do_action(env, act):
    # Thực hiện hành động vật lý lên môi trường thực tế
    x, y = find_location(env)
    if x is None or act == "Noop": return env
    
    new_env = [row[:] for row in env]
    if act == "Suck":
        if new_env[x][y] == 3: new_env[x][y] = 2
        return new_env

    # Tính tọa độ mới dựa trên model
    move = model.get(act, (0, 0))
    nx, ny = x + move[0], y + move[1]

    # Kiểm tra biên và vật cản (-1)
    if 0 <= nx < len(env) and 0 <= ny < len(env[0]) and env[nx][ny] != -1:
        new_env[x][y] = 1 if env[x][y] == 3 else 0
        new_env[nx][ny] = 3 if env[nx][ny] == 1 else 2
    return new_env

In [20]:
def model_based_reflex_agent(percept):
    global state, action, last_pos, visited
    
    curr_pos = find_location(percept)
    
    # Tăng số lần ghé thăm cho ô hiện tại
    if curr_pos and curr_pos[0] is not None:
        visited[curr_pos[0]][curr_pos[1]] += 1
    
    state = update_state(state, action, percept, model)
    new_action = rule_match(state, action, curr_pos, last_pos)
    
    last_pos = curr_pos
    action = new_action
    
    return action

In [21]:
current_env = [row[:] for row in home]
print("Trạng thái ban đầu:")
for row in current_env: print(row)

for i in range(1, 21):
    print(f"\n--- Bước {i} ---")
    act = model_based_reflex_agent(current_env)
    print(f"Hành động quyết định: {act}")
    
    if act == "Noop":
        print("Đã dọn sạch hoặc không còn rác tiếp cận được. Dừng máy!")
        break
        
    current_env = do_action(current_env, act)
    for row in current_env: print(row)

Trạng thái ban đầu:
[1, 1, 3, 1]
[0, -1, 1, 1]
[1, 0, 0, -1]

--- Bước 1 ---
Hành động quyết định: Suck
[1, 1, 2, 1]
[0, -1, 1, 1]
[1, 0, 0, -1]

--- Bước 2 ---
Hành động quyết định: Right
[1, 1, 0, 3]
[0, -1, 1, 1]
[1, 0, 0, -1]

--- Bước 3 ---
Hành động quyết định: Suck
[1, 1, 0, 2]
[0, -1, 1, 1]
[1, 0, 0, -1]

--- Bước 4 ---
Hành động quyết định: Down
[1, 1, 0, 0]
[0, -1, 1, 3]
[1, 0, 0, -1]

--- Bước 5 ---
Hành động quyết định: Suck
[1, 1, 0, 0]
[0, -1, 1, 2]
[1, 0, 0, -1]

--- Bước 6 ---
Hành động quyết định: Left
[1, 1, 0, 0]
[0, -1, 3, 0]
[1, 0, 0, -1]

--- Bước 7 ---
Hành động quyết định: Suck
[1, 1, 0, 0]
[0, -1, 2, 0]
[1, 0, 0, -1]

--- Bước 8 ---
Hành động quyết định: Down
[1, 1, 0, 0]
[0, -1, 0, 0]
[1, 0, 2, -1]

--- Bước 9 ---
Hành động quyết định: Left
[1, 1, 0, 0]
[0, -1, 0, 0]
[1, 2, 0, -1]

--- Bước 10 ---
Hành động quyết định: Left
[1, 1, 0, 0]
[0, -1, 0, 0]
[3, 0, 0, -1]

--- Bước 11 ---
Hành động quyết định: Suck
[1, 1, 0, 0]
[0, -1, 0, 0]
[2, 0, 0, -1]

--- Bước 12